In [ ]:
# Loading the necessary imports
import dynamiqs as dq
import jax.numpy as jnp
from pathlib import Path
import json
import matplotlib.pyplot as plt

In [ ]:
# Building the dictionary of parameters 
params = {
    'Delta_a': -30.0,
    'Delta_b': -30.0,
    'Delta_c': 290.0,

    'Xaa': 0.0014,
    'Xbb': 0.0373,
    'Xcc': 164.0,

    'Xab': 0.0072,
    'Xac': 0.671,
    'Xbc': 3.5,

    'kappa_a': 0.0042,
    'kappa_b': 0.984,
    'kappa_c': 0.0207,

    'Ea': 0.0,
    'Eb': 0.0,
    'Ec': 0.0,

    'Wa' : 4.62e3,
    'wb' : 7.95e3,
    'Wc' : 3.03e3,

    'EJ'   : 7.78e3,
    'phia' : 0.0205,
    'phib' : 0.0468,
    'phic' : 0.453
}

# 1. Add the new drive frequencies (using the original target frequencies)
params['wda'] = params['Wa'] + params['Delta_a']
params['wdb'] = params['wb'] + params['Delta_b']
params['wdc'] = params['Wc'] + params['Delta_c']

# 2. NOW modify the target frequencies to bare frequencies (ZPF shift)
params['Wa'] = params['Wa'] + params['Xaa'] + 0.5 * (params['Xab'] + params['Xac'])
params['Wb'] = params['wb'] + params['Xbb'] + 0.5 * (params['Xab'] + params['Xbc'])
params['Wc'] = params['Wc'] + params['Xcc'] + 0.5 * (params['Xac'] + params['Xbc'])


In [ ]:
# Defining the System Dimensions :
Na = 8
Nb = 8
Nc = 8

# Defining the System Operators :
a = dq.destroy(Na);  a = dq.tensor(a, dq.eye(Nb), dq.eye(Nc))
b = dq.destroy(Nb);  b = dq.tensor(dq.eye(Na), b, dq.eye(Nc))
c = dq.destroy(Nc);  c = dq.tensor(dq.eye(Na), dq.eye(Nb), c)

Numa = a.dag() @ a
Numb = b.dag() @ b
Numc = c.dag() @ c

phi = params['phia']*(a+a.dag()) + params['phib']*(b+b.dag()) + params['phic']*(c+c.dag()) 

# Defining the Hamiltonian, Dissipators and Observables and initial states:
fca = lambda t : jnp.cos(2.0 * jnp.pi * params['wda'] * t)
fsa = lambda t : jnp.sin(2.0 * jnp.pi * params['wda'] * t)

fcb = lambda t : jnp.cos(2.0 * jnp.pi * params['wdb'] * t)
fsb = lambda t : jnp.sin(2.0 * jnp.pi * params['wdb'] * t)

fcc = lambda t : jnp.cos(2.0 * jnp.pi * params['wdc'] * t)
fsc = lambda t : jnp.sin(2.0 * jnp.pi * params['wdc'] * t)

Hda = dq.modulated(fca, a + a.dag()) + 1j * dq.modulated(fsa, a - a.dag())
Hdb = dq.modulated(fcb, b + b.dag()) + 1j * dq.modulated(fsb, b - b.dag())
Hdc = dq.modulated(fcc, c + c.dag()) + 1j * dq.modulated(fsc, c - c.dag())

Hd =  2 * jnp.pi *(params['Ea']*Hda + params['Eb']*Hdb + params['Ec']*Hdc)
H0 = 2 * jnp.pi * (params['Wa']*Numa + params['Wb']*Numb + params['Wc']*Numc - params['EJ']*(dq.cosm(phi) + 0.5*dq.powm(phi,2) ))

# Dissipators :
Dissipators = [jnp.sqrt(params['kappa_a'])*a, jnp.sqrt(params['kappa_b'])*b ,jnp.sqrt(params['kappa_c'])*c,]

# Observables :
Obs = [Numa, Numb, Numc]

# Setting up the Scope of the Simulation :
rho0 = dq.tensor(dq.fock_dm(Na, 0), dq.fock_dm(Nb, 1), dq.fock_dm(Nc, 0))


In [ ]:
# Defining the Scope of the Evolution :
# 1. Combine the Static and Driven Hamiltonian
H = H0 + Hd

# 2. Define the simulation time array (1 microsecond test run)
# Saving 100 points means one data point every 0.01 us (10 ns)
tsave = jnp.linspace(0.0, 1, 101)

# 3. Setup the options (Memory saving ON only)
options = dq.Options(save_states=False)

# 4. Run the Master Equation Solver
print("Starting JAX compilation and 1us GPU execution...")
result = dq.mesolve(
    H, 
    Dissipators, 
    rho0, 
    tsave, 
    exp_ops=Obs, 
    method=dq.method.Tsit5(max_steps=10_000_000),  
    options=options
)
print("Simulation complete!")
